# Chapter 1 Observation-Process and Missingness Notebook

This notebook is a compact visualization layer for the Chapter 1 observation-process variables derived in this repository.

It focuses on the frozen grouped block-centered variables:

- `obs_hr_grp_block`
- `obs_bp_grp_block`
- `obs_resp_grp_block`
- `obs_oxy_grp_block`
- `n_core_grps_obs_block`
- `tsl_hr_grp_h`
- `tsl_bp_grp_h`
- `tsl_resp_grp_h`
- `tsl_oxy_grp_h`

The notebook reads the canonical artifacts already written by the Chapter 1 pipeline and does not re-derive features from raw ASIC inputs.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, Markdown, display
except Exception as exc:  # Notebook execution should normally provide IPython.
    raise RuntimeError(
        "This notebook expects an IPython/Jupyter environment so it can render HTML tables."
    ) from exc

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "artifacts" / "chapter1").exists():
    if (PROJECT_ROOT.parent / "artifacts" / "chapter1").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repo root or the notebooks directory.")

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "chapter1"
OBS_ROOT = ARTIFACT_ROOT / "observation_process"
MODEL_READY_ROOT = ARTIFACT_ROOT / "model_ready"

block_features = pd.read_csv(OBS_ROOT / "chapter1_observation_process_block_features.csv")
qc_summary = pd.read_csv(OBS_ROOT / "chapter1_observation_process_qc_summary.csv")
verification_summary = pd.read_csv(OBS_ROOT / "chapter1_observation_process_verification_summary.csv")
spot_checks = pd.read_csv(OBS_ROOT / "chapter1_observation_process_spot_check_examples.csv")
primary_instances = pd.read_csv(
    MODEL_READY_ROOT / "chapter1_primary_model_ready_with_observation_process.csv"
)
implementation_note = (
    OBS_ROOT / "chapter1_observation_process_implementation_note.md"
).read_text()

OBS_COLS = [
    "obs_hr_grp_block",
    "obs_bp_grp_block",
    "obs_resp_grp_block",
    "obs_oxy_grp_block",
]
TSL_COLS = [
    "tsl_hr_grp_h",
    "tsl_bp_grp_h",
    "tsl_resp_grp_h",
    "tsl_oxy_grp_h",
]
GROUP_LABELS = {
    "obs_hr_grp_block": "HR group",
    "obs_bp_grp_block": "BP group",
    "obs_resp_grp_block": "Respiratory group",
    "obs_oxy_grp_block": "Oxygenation group",
    "tsl_hr_grp_h": "HR group",
    "tsl_bp_grp_h": "BP group",
    "tsl_resp_grp_h": "Respiratory group",
    "tsl_oxy_grp_h": "Oxygenation group",
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [2]:
PALETTE = {
    "teal": "#0f766e",
    "blue": "#1d4ed8",
    "orange": "#c2410c",
    "gold": "#ca8a04",
    "slate": "#475569",
}


def show_markdown(text: str) -> None:
    display(Markdown(text))


def format_value(value: object, formatter: object) -> str:
    if pd.isna(value):
        return "missing"
    if callable(formatter):
        return str(formatter(value))
    return str(formatter.format(value))


def bar_html(
    value: object,
    *,
    scale: float,
    color: str,
    formatter: object,
    width_px: int = 160,
) -> str:
    label = format_value(value, formatter)
    if pd.isna(value) or scale <= 0:
        filled_width = 0.0
    else:
        filled_width = max(0.0, min(width_px, width_px * float(value) / float(scale)))
    return (
        f"<div style='display:flex;align-items:center;gap:10px;'>"
        f"<div style='width:{width_px}px;height:14px;background:#e5e7eb;border-radius:999px;overflow:hidden;'>"
        f"<div style='width:{filled_width:.1f}px;height:14px;background:{color};'></div>"
        f"</div>"
        f"<span style='font-family:monospace;font-size:12px;'>{label}</span>"
        f"</div>"
    )


def html_bar_table(
    df: pd.DataFrame,
    *,
    bar_columns: list[str],
    scales: dict[str, float] | None = None,
    formatters: dict[str, object] | None = None,
    colors: dict[str, str] | None = None,
    index: bool = False,
) -> HTML:
    scales = scales or {}
    formatters = formatters or {}
    colors = colors or {}
    rendered = df.copy()
    for column in bar_columns:
        numeric = pd.to_numeric(df[column], errors="coerce")
        scale = scales.get(column)
        if scale is None:
            scale = float(numeric.max()) if numeric.notna().any() else 0.0
        formatter = formatters.get(column, "{:.2f}")
        color = colors.get(column, PALETTE["teal"])
        rendered[column] = [
            bar_html(value, scale=scale, color=color, formatter=formatter)
            for value in df[column]
        ]
    return HTML(rendered.to_html(index=index, escape=False))


def stale_bucket(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    buckets = pd.Series("missing", index=series.index, dtype="string")
    buckets.loc[numeric.le(1)] = "<=1h"
    buckets.loc[numeric.gt(1) & numeric.lt(8)] = "1-8h"
    buckets.loc[numeric.ge(8) & numeric.lt(24)] = "8-24h"
    buckets.loc[numeric.ge(24)] = ">=24h"
    return buckets


## Artifact Overview

Use `block_features` for unique usable 8-hour blocks before horizon duplication.
Use `primary_instances` for the horizon-specific analysis-ready table after label filtering and split assignment.


In [3]:
artifact_overview = pd.DataFrame(
    [
        {
            "artifact": "observation_process_block_features",
            "rows": int(block_features.shape[0]),
            "columns": int(block_features.shape[1]),
            "meaning": "Unique usable 8-hour blocks before horizon duplication.",
        },
        {
            "artifact": "primary_model_ready_with_observation_process",
            "rows": int(primary_instances.shape[0]),
            "columns": int(primary_instances.shape[1]),
            "meaning": "Analysis-ready rows after horizon duplication, label filtering, and split annotation.",
        },
        {
            "artifact": "observation_process_verification_summary",
            "rows": int(verification_summary.shape[0]),
            "columns": int(verification_summary.shape[1]),
            "meaning": "Requested derivation checks for binary indicators, grouped completeness, and non-negative `tsl_*`.",
        },
    ]
)
display(artifact_overview)

show_markdown("## Derivation Contract")
show_markdown(implementation_note)

show_markdown("## Verification Checks")
display(verification_summary)


,artifact,rows,columns,meaning
0,observation_process_block_features,1328,15,Unique usable 8-hour blocks before horizon dup...
1,primary_model_ready_with_observation_process,6080,327,"Analysis-ready rows after horizon duplication,..."
2,observation_process_verification_summary,18,4,Requested derivation checks for binary indicat...


## Derivation Contract

# Chapter 1 Observation-Process Variable Note

## Final Group Mapping Used
- `HR group`: `heart_rate`
- `BP group`: `map`, `sbp`, `dbp`
- `Respiratory group`: `resp_rate`
- `Oxygenation group`: `spo2`, `sao2`

## Derivation Rules
- Input source: raw `dynamic/harmonized` ASIC measurements plus unique usable 8-hour blocks.
- Block membership uses the existing upstream ASIC blocked-data contract: raw measurements with `minutes_since_admit` in `[block_start_h, block_end_h)`.
- `obs_*_grp_block` equals 1 when any raw group measurement is observed inside the current 8-hour block; otherwise 0.
- `n_core_grps_obs_block` is the row-wise sum of the four binary group indicators.
- `tsl_*` equals `prediction_time_h - latest_raw_observed_time_h` within stay and group, using all raw history with `minutes_since_admit < prediction_time_h * 60`.
- If a group has never been observed up to prediction time, the corresponding `tsl_*` stays missing.

## Deviations From Requested Design
- None in the derived variables themselves.
- The block-level export is unique per usable 8-hour block before horizon duplication; separate optional merged model-ready artifacts duplicate the block features across horizon-specific prediction-instance rows for convenience.

## Limitations / Ambiguities
- The raw-history derivation depends on `minutes_since_admit` in the harmonized ASIC dynamic table being aligned with the blocked 8-hour artifacts.
- Exact boundary handling was chosen to match the upstream blocked `*_obs_count` contract and verified empirically against the blocked feature table.

## Verification Checks

,check_id,passed,value,detail
0,obs_hr_grp_block_binary_values_only,True,1,Observed-in-block indicators must be binary {0...
1,obs_bp_grp_block_binary_values_only,True,"0, 1",Observed-in-block indicators must be binary {0...
2,obs_resp_grp_block_binary_values_only,True,"0, 1",Observed-in-block indicators must be binary {0...
3,obs_oxy_grp_block_binary_values_only,True,"0, 1",Observed-in-block indicators must be binary {0...
4,n_core_grps_obs_block_matches_group_sum,True,1328,n_core_grps_obs_block must equal the row-wise ...
5,n_core_grps_obs_block_within_expected_range,True,1328,n_core_grps_obs_block must always fall between...
6,tsl_hr_grp_h_non_negative,True,1328,Time-since-last values must be non-negative.
7,tsl_hr_grp_h_never_observed_rows_remain_missing,True,0,Rows with no raw observation history in the gr...
8,tsl_hr_grp_h_current_block_recent_examples_pre...,True,1328,At least one row should show an in-block obser...
9,tsl_bp_grp_h_non_negative,True,1166,Time-since-last values must be non-negative.


## Overall Block-Level Views

These summaries use the unique block-level table so that each completed 8-hour block contributes once.


In [4]:
overall_observed = pd.DataFrame(
    {
        "group": [GROUP_LABELS[column] for column in OBS_COLS],
        "observed_blocks": [int(block_features[column].sum()) for column in OBS_COLS],
        "observed_rate": [float(block_features[column].mean()) for column in OBS_COLS],
    }
)
display(
    html_bar_table(
        overall_observed,
        bar_columns=["observed_blocks", "observed_rate"],
        scales={"observed_rate": 1.0},
        formatters={"observed_blocks": "{:.0f}", "observed_rate": "{:.1%}"},
        colors={"observed_blocks": PALETTE["teal"], "observed_rate": PALETTE["blue"]},
    )
)

n_core_distribution = (
    block_features["n_core_grps_obs_block"]
    .value_counts(dropna=False)
    .reindex(range(5), fill_value=0)
    .rename_axis("n_core_grps_obs_block")
    .reset_index(name="block_count")
)
n_core_distribution["block_share"] = (
    n_core_distribution["block_count"] / max(len(block_features), 1)
)
display(
    html_bar_table(
        n_core_distribution,
        bar_columns=["block_count", "block_share"],
        scales={"block_share": 1.0},
        formatters={"block_count": "{:.0f}", "block_share": "{:.1%}"},
        colors={"block_count": PALETTE["orange"], "block_share": PALETTE["gold"]},
    )
)

tsl_availability = pd.DataFrame(
    {
        "group": [GROUP_LABELS[column] for column in TSL_COLS],
        "non_missing_rows": [int(block_features[column].notna().sum()) for column in TSL_COLS],
        "never_observed_rows": [int(block_features[column].isna().sum()) for column in TSL_COLS],
        "non_missing_rate": [float(block_features[column].notna().mean()) for column in TSL_COLS],
    }
)
display(
    html_bar_table(
        tsl_availability,
        bar_columns=["non_missing_rows", "never_observed_rows", "non_missing_rate"],
        scales={"non_missing_rate": 1.0},
        formatters={
            "non_missing_rows": "{:.0f}",
            "never_observed_rows": "{:.0f}",
            "non_missing_rate": "{:.1%}",
        },
        colors={
            "non_missing_rows": PALETTE["teal"],
            "never_observed_rows": PALETTE["orange"],
            "non_missing_rate": PALETTE["blue"],
        },
    )
)


group,observed_blocks,observed_rate
HR group,1328,100.0%
BP group,1127,84.9%
Respiratory group,1259,94.8%
Oxygenation group,1326,99.8%


n_core_grps_obs_block,block_count,block_share
0,0,0.0%
1,0,0.0%
2,0,0.0%
3,272,20.5%
4,1056,79.5%


group,non_missing_rows,never_observed_rows,non_missing_rate
HR group,1328,0,100.0%
BP group,1166,162,87.8%
Respiratory group,1324,4,99.7%
Oxygenation group,1328,0,100.0%


## Horizon and Split Views

These summaries use the primary analysis-ready table so the observation-process variables can be inspected in the same horizon / split frame that later sensitivity analyses will use.


In [5]:
observed_by_horizon = (
    primary_instances.groupby("horizon_h", dropna=False)[OBS_COLS]
    .mean()
    .rename(columns=GROUP_LABELS)
    .reset_index()
)
display(
    html_bar_table(
        observed_by_horizon,
        bar_columns=[column for column in observed_by_horizon.columns if column != "horizon_h"],
        scales={column: 1.0 for column in observed_by_horizon.columns if column != "horizon_h"},
        formatters={column: "{:.1%}" for column in observed_by_horizon.columns if column != "horizon_h"},
        colors={
            "HR group": PALETTE["teal"],
            "BP group": PALETTE["blue"],
            "Respiratory group": PALETTE["orange"],
            "Oxygenation group": PALETTE["gold"],
        },
    )
)

mean_core_by_split_horizon = (
    primary_instances.pivot_table(
        index="horizon_h",
        columns="split",
        values="n_core_grps_obs_block",
        aggfunc="mean",
    )
    .reindex(columns=[column for column in ["train", "validation", "test"] if column in primary_instances["split"].unique()])
    .reset_index()
)
display(
    html_bar_table(
        mean_core_by_split_horizon,
        bar_columns=[column for column in mean_core_by_split_horizon.columns if column != "horizon_h"],
        scales={column: 4.0 for column in mean_core_by_split_horizon.columns if column != "horizon_h"},
        formatters={column: "{:.2f}" for column in mean_core_by_split_horizon.columns if column != "horizon_h"},
        colors={
            "train": PALETTE["teal"],
            "validation": PALETTE["blue"],
            "test": PALETTE["orange"],
        },
    )
)

label_stratified = (
    primary_instances.groupby(["horizon_h", "label_value"], dropna=False)
    .agg(
        rows=("instance_id", "size"),
        mean_n_core_grps_obs_block=("n_core_grps_obs_block", "mean"),
        share_all_4_groups=("n_core_grps_obs_block", lambda series: float(series.eq(4).mean())),
    )
    .reset_index()
)
label_stratified["label"] = label_stratified["label_value"].map({0: "negative", 1: "positive"})
label_stratified = label_stratified[[
    "horizon_h",
    "label",
    "rows",
    "mean_n_core_grps_obs_block",
    "share_all_4_groups",
]]
display(
    html_bar_table(
        label_stratified,
        bar_columns=["rows", "mean_n_core_grps_obs_block", "share_all_4_groups"],
        scales={"mean_n_core_grps_obs_block": 4.0, "share_all_4_groups": 1.0},
        formatters={
            "rows": "{:.0f}",
            "mean_n_core_grps_obs_block": "{:.2f}",
            "share_all_4_groups": "{:.1%}",
        },
        colors={
            "rows": PALETTE["slate"],
            "mean_n_core_grps_obs_block": PALETTE["teal"],
            "share_all_4_groups": PALETTE["gold"],
        },
    )
)


horizon_h,HR group,BP group,Respiratory group,Oxygenation group
8,100.0%,84.6%,95.5%,99.8%
16,100.0%,84.6%,95.6%,99.8%
24,100.0%,84.6%,95.6%,99.8%
48,100.0%,84.1%,95.7%,99.8%
72,100.0%,83.5%,95.7%,99.8%


split,horizon_h,train,validation,test
,8,3.89,3.34,3.98
,16,3.89,3.34,3.98
,24,3.89,3.34,3.98
,48,3.89,3.33,3.98
,72,3.89,3.34,3.98


horizon_h,label,rows,mean_n_core_grps_obs_block,share_all_4_groups
8,negative,1246,3.80,80.2%
8,positive,10,3.60,60.0%
16,negative,1224,3.80,80.3%
16,positive,20,3.65,65.0%
24,negative,1202,3.80,80.4%
24,positive,30,3.67,66.7%
48,negative,1136,3.80,80.0%
48,positive,58,3.72,72.4%
72,negative,1072,3.80,79.8%
72,positive,82,3.70,69.5%


## Time-Since-Last Measurement

The `tsl_*` variables are displayed both as summary statistics and as coarse staleness buckets.
Missing values here correspond to groups that were never observed up to prediction time.


In [6]:
tsl_quantiles = []
for column in TSL_COLS:
    series = pd.to_numeric(block_features[column], errors="coerce")
    tsl_quantiles.append(
        {
            "group": GROUP_LABELS[column],
            "non_missing_rows": int(series.notna().sum()),
            "never_observed_rows": int(series.isna().sum()),
            "median_h": float(series.median()) if series.notna().any() else np.nan,
            "p75_h": float(series.quantile(0.75)) if series.notna().any() else np.nan,
            "p90_h": float(series.quantile(0.90)) if series.notna().any() else np.nan,
            "max_h": float(series.max()) if series.notna().any() else np.nan,
        }
    )

tsl_quantiles = pd.DataFrame(tsl_quantiles)
display(
    html_bar_table(
        tsl_quantiles,
        bar_columns=["non_missing_rows", "never_observed_rows", "median_h", "p75_h", "p90_h", "max_h"],
        formatters={
            "non_missing_rows": "{:.0f}",
            "never_observed_rows": "{:.0f}",
            "median_h": "{:.2f}",
            "p75_h": "{:.2f}",
            "p90_h": "{:.2f}",
            "max_h": "{:.2f}",
        },
        colors={
            "non_missing_rows": PALETTE["teal"],
            "never_observed_rows": PALETTE["orange"],
            "median_h": PALETTE["blue"],
            "p75_h": PALETTE["blue"],
            "p90_h": PALETTE["gold"],
            "max_h": PALETTE["slate"],
        },
    )
)

bucket_rows = []
for column in TSL_COLS:
    bucket_distribution = stale_bucket(block_features[column]).value_counts(normalize=True)
    bucket_rows.append(
        {
            "group": GROUP_LABELS[column],
            "<=1h": float(bucket_distribution.get("<=1h", 0.0)),
            "1-8h": float(bucket_distribution.get("1-8h", 0.0)),
            "8-24h": float(bucket_distribution.get("8-24h", 0.0)),
            ">=24h": float(bucket_distribution.get(">=24h", 0.0)),
            "missing": float(bucket_distribution.get("missing", 0.0)),
        }
    )

staleness_buckets = pd.DataFrame(bucket_rows)
display(
    html_bar_table(
        staleness_buckets,
        bar_columns=["<=1h", "1-8h", "8-24h", ">=24h", "missing"],
        scales={column: 1.0 for column in ["<=1h", "1-8h", "8-24h", ">=24h", "missing"]},
        formatters={column: "{:.1%}" for column in ["<=1h", "1-8h", "8-24h", ">=24h", "missing"]},
        colors={
            "<=1h": PALETTE["teal"],
            "1-8h": PALETTE["blue"],
            "8-24h": PALETTE["gold"],
            ">=24h": PALETTE["orange"],
            "missing": PALETTE["slate"],
        },
    )
)


group,non_missing_rows,never_observed_rows,median_h,p75_h,p90_h,max_h
HR group,1328,0,0.25,0.50,1.00,6.00
BP group,1166,162,0.25,0.25,1.50,166.75
Respiratory group,1324,4,0.25,0.75,2.25,301.50
Oxygenation group,1328,0,0.25,0.50,1.25,21.50


group,<=1h,1-8h,8-24h,>=24h,missing
HR group,90.7%,9.3%,0.0%,0.0%,0.0%
BP group,76.1%,8.7%,0.6%,2.3%,12.2%
Respiratory group,79.3%,15.5%,2.1%,2.8%,0.3%
Oxygenation group,89.9%,9.9%,0.2%,0.0%,0.0%


## Spot Checks

These are the concrete examples written by the derivation step to sanity-check current-block observations, prior-history-only measurements, and never-observed cases.


In [7]:
display(spot_checks)


,group_name,scenario,stay_id_global,hospital_id,block_index,block_start_h,block_end_h,prediction_time_h,obs_indicator_column,obs_indicator_value,time_since_last_column,time_since_last_hours,latest_raw_observed_h
0,cardiac_rate,observed_in_current_block_recent_measurement,asic_UK02_9990,asic_UK02,0,0,8,8,obs_hr_grp_block,1,tsl_hr_grp_h,0.25,7.75
1,blood_pressure,observed_in_current_block_recent_measurement,asic_UK02_9990,asic_UK02,0,0,8,8,obs_bp_grp_block,1,tsl_bp_grp_h,0.25,7.75
2,blood_pressure,historical_measurement_before_current_block,asic_UK02_9991,asic_UK02,52,416,424,424,obs_bp_grp_block,0,tsl_bp_grp_h,12.00,412.00
3,blood_pressure,never_observed_by_prediction_time,asic_UK02_9999,asic_UK02,0,0,8,8,obs_bp_grp_block,0,tsl_bp_grp_h,NaN,NaN
4,respiratory,observed_in_current_block_recent_measurement,asic_UK02_9990,asic_UK02,0,0,8,8,obs_resp_grp_block,1,tsl_resp_grp_h,2.25,5.75
5,respiratory,historical_measurement_before_current_block,asic_UK04_9992,asic_UK04,3,24,32,32,obs_resp_grp_block,0,tsl_resp_grp_h,18.25,13.75
6,respiratory,never_observed_by_prediction_time,asic_UK07_9990,asic_UK07,5,40,48,48,obs_resp_grp_block,0,tsl_resp_grp_h,NaN,NaN
7,oxygenation,observed_in_current_block_recent_measurement,asic_UK02_9990,asic_UK02,0,0,8,8,obs_oxy_grp_block,1,tsl_oxy_grp_h,0.25,7.75
8,oxygenation,historical_measurement_before_current_block,asic_UK02_9990,asic_UK02,3,24,32,32,obs_oxy_grp_block,0,tsl_oxy_grp_h,13.50,18.50
